## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-15-07-10-31 +0000,bbc,Copper wire upgrade behind US flight suspensio...,https://www.bbc.co.uk/news/articles/c6wy9lynq4...
1,2026-11-15-06-41-42 +0000,nypost,Federal judge blocks Trump from moving ahead o...,https://nypost.com/2026/11/15/us-news/trump-bl...
2,2026-11-15-06-27-05 +0000,nypost,"Turkish husband, who pushed pregnant wife off ...",https://nypost.com/2026/11/15/world-news/turki...
3,2026-11-15-05-51-30 +0000,nypost,Bahrain issues first threat alerts since Septe...,https://nypost.com/2026/11/15/world-news/iran-...
4,2026-11-15-05-47-32 +0000,nyt,Two Planes Collide on a Taxiway at Miami Inter...,https://www.nytimes.com/2026/11/15/us/miami-ai...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2432/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
10,trump,18
243,ground,14
145,california,14
290,new,13
31,first,11


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
157,2026-11-14-15-53-55 +0000,bbc,"Trump, Putin and Xi plan historic joint meetin...",https://www.bbc.co.uk/news/articles/ckz74jzpy1...,62
31,2026-11-15-02-53-00 +0000,wsj,A federal judge temporarily barred the Trump a...,https://www.wsj.com/politics/policy/trump-admi...,58
1,2026-11-15-06-41-42 +0000,nypost,Federal judge blocks Trump from moving ahead o...,https://nypost.com/2026/11/15/us-news/trump-bl...,53
114,2026-11-14-20-08-07 +0000,cbc,New Orleans port sues for control of China-lin...,https://www.cbc.ca/news/world/new-orleans-tank...,52
170,2026-11-14-14-01-46 +0000,nypost,Trump weighs holding off on new Iran strikes u...,https://nypost.com/2026/11/14/world-news/presi...,50


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
157,62,2026-11-14-15-53-55 +0000,bbc,"Trump, Putin and Xi plan historic joint meetin...",https://www.bbc.co.uk/news/articles/ckz74jzpy1...
48,48,2026-11-15-01-02-12 +0000,nypost,Tanker that struck New Orleans cruise ship had...,https://nypost.com/2026/11/14/us-news/tanker-t...
179,48,2026-11-14-13-10-00 +0000,wsj,Weekend Air Travel Woes Mount After FAA Lifts ...,https://www.wsj.com/us-news/faa-temporarily-ha...
143,44,2026-11-14-17-27-50 +0000,nypost,Musk wants disgraced ex-Nvidia boss Jensen Hua...,https://nypost.com/2026/11/14/world-news/elon-...
87,41,2026-11-14-22-15-52 +0000,bbc,SpaceX successfully catches top of huge Starsh...,https://www.bbc.co.uk/news/articles/ckregdgnl0...
31,40,2026-11-15-02-53-00 +0000,wsj,A federal judge temporarily barred the Trump a...,https://www.wsj.com/politics/policy/trump-admi...
67,39,2026-11-14-23-26-39 +0000,nypost,California’s AI craze hits the road as Silicon...,https://nypost.com/2026/11/14/us-news/san-jose...
21,32,2026-11-15-03-27-33 +0000,nypost,Hundreds gather to remember ‘loving’ young con...,https://nypost.com/2026/11/14/us-news/hundreds...
132,32,2026-11-14-18-35-00 +0000,wsj,Transportation Secretary Sean Duffy blames Con...,https://www.wsj.com/politics/policy/duffy-blam...
147,31,2026-11-14-17-00-08 +0000,chitri,Microsoft says services fully restored after h...,https://www.chicagotribune.com/2026/11/14/micr...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
